# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore a Croissant-structured biomedical dataset using the `mlcroissant` library.

### Dataset Source
The Croissant dataset is sourced from the following schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed in this environment.
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and prepare for record exploration.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant metadata URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata_obj = dataset.metadata
print(f"{metadata_obj.name}: {metadata_obj.description}")

## 2. Data Overview
Let's inspect the available record sets in the dataset. Each record set, field, and column is referenced by its Croissant `@id`.

**Note:** We use `dataset.metadata.record_sets` for programmatic access to Record Sets. Each Field and Column will also be referenced by `@id`.

In [ ]:
# List all record sets, fields, and column IDs
record_sets = dataset.metadata.record_sets
print(f"Found {len(record_sets)} record sets in this dataset.\n")
for rs in record_sets:
    print(f"Record Set: {rs.name}")
    print(f"  @id: {rs.id}")
    print(f"  Fields:")
    for field in rs.fields:
        columns = getattr(field, 'columns', [])
        col_str = ', '.join([c.id for c in columns]) if columns else "[No columns]"
        print(f"    - {field.name} (@id: {field.id}) Columns: {col_str}")
    print()

## 3. Data Extraction
We'll extract records from the primary record set (table) for analysis. In this dataset, you generally want the record set that contains the mass-spectrometry protein table. We'll list all defined record set `@id`s for convenience.

_Note: Fields (columns) will use their `@id` in DataFrame operations._

In [ ]:
# Collect Record Set @ids
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
print("Record Set IDs:", record_set_ids)

# For demonstration, choose the first record set
primary_record_set_id = record_set_ids[0]

# Load records from each record set into a DataFrame
dataframes = {}
for rsid in record_set_ids:
    records = list(dataset.records(record_set=rsid))
    df = pd.DataFrame(records)
    dataframes[rsid] = df
    print(f"Loaded {len(df)} records from record set: {rsid}")

# Display columns for the main table
print(f"Columns for {primary_record_set_id}:")
print(dataframes[primary_record_set_id].columns.tolist())

# Show the first 5 rows
dataframes[primary_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)

Process, filter, and explore the dataset numerically and categorically.

Below we:
- Select a numeric field (by inspecting available columns)
- Filter based on a threshold
- Normalize data
- Optionally group by a categorical field (if available)

_Please adjust the values to suitable field `@id`s for your use case as needed._

In [ ]:
# Choose a numeric field (by @id) for analysis; update if needed
df = dataframes[primary_record_set_id]
print("Available columns:", df.columns.tolist())

# Example numeric field (guessing presence from protein datasets, e.g., MW or abundance)
# You should update these IDs based on the previous print of columns
possible_numeric_fields = [col for col in df.columns if df[col].dtype in [float, int] or pd.api.types.is_numeric_dtype(df[col])]
print("Numeric fields candidates:", possible_numeric_fields)

# For illustration, select the first numeric field
if possible_numeric_fields:
    numeric_field_id = possible_numeric_fields[0]
else:
    # If columns are object type, try to coerce one to numeric
    numeric_field_id = df.columns[0]
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors="coerce")

threshold = 10  # Example threshold (update for your field)
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold}:")
print(filtered_df.head())

# Normalize the field
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id}:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Try grouping by a likely categorical field
cat_fields = [col for col in df.columns if df[col].dtype == object]
group_field = cat_fields[0] if cat_fields else None

if group_field and group_field in df.columns:
    grouped = filtered_df.groupby(group_field)[numeric_field_id].mean()
    print(f"Grouped data by {group_field} (mean {numeric_field_id}):")
    print(grouped.head())

## 5. Visualization

Let's plot the distribution of the selected numeric field and relationships with a categorical field (if present).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of numeric field
plt.figure(figsize=(7,4))
sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
plt.title(f"Histogram of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.show()

# Boxplot by categorical field if available
if group_field and group_field in df.columns:
    plt.figure(figsize=(10, 5))
    sns.boxplot(x=group_field, y=numeric_field_id, data=df)
    plt.title(f"{numeric_field_id} by {group_field}")
    plt.xticks(rotation=45)
    plt.show()


## 6. Conclusion

In this notebook, we demonstrated how to use `mlcroissant` with a Croissant-described mass spectrometry dataset. We reviewed available record sets and fields, loaded data dynamically by referenced `@id`s, performed basic filtering and normalization, and visualized value distributions. 

Next steps could include deeper feature engineering, machine learning modeling, or domain-specific analyses on the protein data.